In [177]:
## Upload File (Load Dataset)&Extract Text (Convert Dataset → Text)

In [178]:
pip install PyPDF2 sentence-transformers faiss-cpu numpy

Note: you may need to restart the kernel to use updated packages.


In [179]:
import PyPDF2
pdf_path=r"C:\Users\SOURADEEP CHOWDHURY\Downloads\HaenleinMichael-ABriefHistoryofArtificialIntelligence.pdf"
reader=PyPDF2.PdfReader(pdf_path)
text=""
for page in reader.pages:
    t=page.extract_text()
    if t:
        text +=t + ""
print(len(text))

32612


In [180]:
## Split Text into Chunks

In [181]:
chunk_size=800
chunks=[]
for i in range(0,len(text),chunk_size):
    chunks.append(text[i:i+chunk_size])
print("Total chunks:",len(chunks))

Total chunks: 41


In [182]:
## Generate Embeddings for Each Chunk

In [183]:
from sentence_transformers import SentenceTransformer 

In [184]:
model=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5705.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [185]:
embeddings=model.encode(chunks)


In [186]:
print(embeddings.shape)

(41, 384)


In [187]:
## Store Embeddings in FAISS

In [ ]:
import faiss
import numpy as np
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print('Stored:',index.ntotal)

Stored: 41


In [189]:
## User Inputs a Question

In [251]:
query="how the Artificial neural networks made a comeback?"

In [252]:
## Convert Question to Embedding

In [253]:
query_embeddings=model.encode([query])

In [254]:
query_embeddings

array([[-8.81078690e-02, -7.28776753e-02,  7.35465139e-02,
         3.94297689e-02,  1.52877634e-02,  5.35479523e-02,
        -3.25688049e-02,  3.91963944e-02, -1.12615321e-02,
        -5.20185195e-02, -1.58175863e-02,  1.13802031e-01,
         1.53174251e-02, -5.80755025e-02, -1.19575672e-01,
        -2.44866125e-02, -6.43533692e-02,  3.33665088e-02,
        -8.22937861e-02, -5.00840656e-02, -8.51223096e-02,
         1.64105520e-02, -4.21339348e-02, -7.25442776e-03,
         6.29151464e-02,  7.90445134e-02, -3.61973047e-02,
        -1.97302084e-02,  4.39460110e-03, -1.88605934e-02,
         2.32526083e-02,  1.66398007e-02,  2.08951589e-02,
         3.01510096e-03, -1.10569164e-01,  3.83583568e-02,
         4.43668861e-04,  1.37140378e-02,  5.46759181e-03,
        -8.88729352e-04, -3.57130542e-02, -8.34709778e-02,
        -1.44833745e-02, -4.82136421e-02,  9.89921540e-02,
         2.63218656e-02,  3.60567048e-02, -3.43979821e-02,
         2.74599735e-02,  1.85498614e-02, -4.91806529e-0

In [255]:
## Retrieve Top Relevant Chunks

In [256]:
k = 2

distances, indices = index.search(query_embeddings, k)

In [257]:
## Return Retrieved Text as Response

In [258]:
idx = indices[0][0]

chunk_text = chunks[idx]

sentences = chunk_text.split(".")

sent_embeddings = model.encode(sentences)

dim = sent_embeddings.shape[1]
sent_index = faiss.IndexFlatL2(dim)
sent_index.add(np.array(sent_embeddings))

q_vec = model.encode([query])

D, I = sent_index.search(q_vec, 1)

answer_sentence = sentences[I[0][0]]

print("Retrieved Chunk:\n", chunk_text[:500])

Retrieved Chunk:
  such data, and to use those learnings to achieve specific goals and tasks through flexible adaptation—charac-teristics that define AI.
9 Since Expert Systems do not possess these characteristics, 
they are technically speaking not true AI. Statistical methods for achieving true AI have been discussed as early as the 1940s when the Canadian psychologist Donald Hebb developed a theory of learning known as Hebbian Learning that replicates the process of neurons in the human brain.
10 This led to t


In [259]:
k = 3
distances, indices = index.search(model.encode([query]), k)

for i in indices[0]:
    print(chunks[i][:400])

 such data, and to use those learnings to achieve specific goals and tasks through flexible adaptation—charac-teristics that define AI.
9 Since Expert Systems do not possess these characteristics, 
they are technically speaking not true AI. Statistical methods for achieving true AI have been discussed as early as the 1940s when the Canadian psychologist Donald Hebb developed a theory of learning k
 although the Japanese 
government began to heavily fund AI research in the 1980s, to which the U.S. DARPA responded by a funding increase as well, no further advances were made in the following years.
AI Fall: The Harvest
One reason for the initial lack of progress in the field of AI and the fact 
that reality fell back sharply relative to expectations lies in the specific way in which early syst
in the form of Deep Learning 
when in 2015 AlphaGo, a program developed by Google, was able to beat the world champion in the board game Go. Go is substantially more complex than chess (e.g., at ope

In [260]:
query = "Who proposed Hebbian Learning?"

In [261]:
k = 3
distances, indices = index.search(model.encode([query]), k)

In [263]:
idx = indices[0][0]

chunk_text = chunks[idx]

sentences = chunk_text.split(".")

sent_embeddings = model.encode(sentences)

dim = sent_embeddings.shape[1]
sent_index = faiss.IndexFlatL2(dim)
sent_index.add(np.array(sent_embeddings))

q_vec = model.encode([query])

D, I = sent_index.search(q_vec, 1)

answer_sentence = sentences[I[0][0]]

print(answer_sentence)

 Statistical methods for achieving true AI have been discussed as early as the 1940s when the Canadian psychologist Donald Hebb developed a theory of learning known as Hebbian Learning that replicates the process of neurons in the human brain
